# Classification leaderboard (Phase 1 & Phase 2 comparison)

Loads all `*_metrics.json` artifacts from `outputs/models/clf/`, builds a leaderboard table (target-family × model × metric × split), plots confusion matrices and confident-accuracy-vs-coverage curves, and saves:
- `outputs/reports/phase1_leaderboard.csv` — 1h-grid results
- `outputs/reports/phase2_representation_leaderboard.csv` — event-bar results (when available)

**Prerequisites:** run all combos via `make train-clf-sweep` or run each `ClassifierRunner` call in `experiment_tracking.ipynb`.

In [ ]:
import sys, json, pathlib, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, str(pathlib.Path('.').resolve()))

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

CLF_DIR     = pathlib.Path('outputs/models/clf')
REPORTS_DIR = pathlib.Path('outputs/reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load all metrics.json files

In [ ]:
def load_all_metrics(base_dir: pathlib.Path) -> pd.DataFrame:
    rows = []
    for metrics_file in sorted(base_dir.rglob('*_metrics.json')):
        try:
            m = json.loads(metrics_file.read_text())
            # Flatten for both CLF and regression formats
            if 'metrics' not in m:
                continue
            cfg = m.get('config', {})
            label_cfg = cfg.get('label', {})
            if isinstance(label_cfg, dict):
                family = label_cfg.get('target_family', metrics_file.parent.name.split('_', 1)[-1])
            else:
                family = metrics_file.parent.name.split('_', 1)[-1]
            model_name = cfg.get('model_name', metrics_file.parent.name.split('_')[0])
            data_csv   = cfg.get('data_csv', 'unknown')
            representation = 'event_volume' if 'volume' in str(data_csv) else \
                             'event_dollar' if 'dollar' in str(data_csv) else '1h_grid'

            for split in ('val', 'test'):
                sm = m['metrics'].get(split, {})
                if not sm:
                    continue
                bl = m.get('baselines', {}).get(split, {}).get('majority_class', {})
                rows.append({
                    'model':           model_name,
                    'target_family':   family,
                    'representation':  representation,
                    'split':           split,
                    'balanced_acc':    sm.get('balanced_accuracy'),
                    'mcc':             sm.get('mcc'),
                    'roc_auc':         sm.get('roc_auc'),
                    'pr_auc':          sm.get('pr_auc'),
                    'log_loss':        sm.get('log_loss'),
                    'confident_acc':   sm.get('confident_accuracy'),
                    'trading_coverage':sm.get('trading_coverage'),
                    'trading_hit_rate':sm.get('trading_hit_rate'),
                    'trading_sharpe':  sm.get('trading_sharpe_annual'),
                    'n_samples':       sm.get('n_samples'),
                    'baseline_bal_acc': bl.get('balanced_accuracy'),
                    'source_file':     str(metrics_file),
                })
        except Exception as e:
            print(f'Skipping {metrics_file}: {e}')
    return pd.DataFrame(rows)

df_all = load_all_metrics(pathlib.Path('outputs/models'))
print(f'Loaded {len(df_all)} rows from {df_all["source_file"].nunique()} metrics files')
df_all[['model', 'target_family', 'representation', 'split', 'balanced_acc', 'mcc', 'roc_auc']].head(10)

## 2. Phase 1 leaderboard (1h-grid, test split)

In [ ]:
p1 = df_all[
    (df_all['representation'] == '1h_grid') & (df_all['split'] == 'test')
].copy()

if len(p1) == 0:
    print('No Phase 1 results found. Run: make train-clf-sweep')
else:
    leaderboard = p1.sort_values('roc_auc', ascending=False)[
        ['model', 'target_family', 'balanced_acc', 'mcc', 'roc_auc',
         'trading_coverage', 'trading_hit_rate', 'trading_sharpe', 'baseline_bal_acc']
    ].reset_index(drop=True)
    leaderboard.index += 1

    leaderboard.to_csv(REPORTS_DIR / 'phase1_leaderboard.csv')
    print(f'Saved → outputs/reports/phase1_leaderboard.csv')
    print(leaderboard.to_string())

## 3. Val vs Test consistency check (no single-split flukes)

In [ ]:
p1_pivot = df_all[
    (df_all['representation'] == '1h_grid') &
    df_all['target_family'].isin(['vol_regime', 'large_move', 'direction'])
].pivot_table(
    index=['model', 'target_family'], columns='split',
    values=['balanced_acc', 'roc_auc']
).round(4)

print('Val vs Test consistency:')
print(p1_pivot.to_string())

## 4. Confusion matrices (test set, all families)

In [ ]:
from chronos_ts.labels import LabelConfig, LabelMaker
from chronos_ts.splits import TimeRangeSplitConfig, time_fraction_split
import numpy as np

# Load the label makers to get class names
df_data = pd.read_csv('outputs/datasets/btcusdt_clf_core.csv', parse_dates=['ts'])
df_data = df_data.sort_values('ts').reset_index(drop=True)
splits = time_fraction_split(df_data, TimeRangeSplitConfig(), ts_col='ts')

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.ravel()
ax_idx = 0

families = ['direction', 'large_move', 'vol_regime', 'horizon_dir', 'return_token']
models   = ['logreg', 'catboost']

for model in models:
    for family in families:
        pred_file = pathlib.Path(f'outputs/models/clf/{model}_{family}/{model}_{family}_test_predictions.csv')
        if not pred_file.exists():
            axes[ax_idx].text(0.5, 0.5, 'Not run yet', ha='center', va='center', transform=axes[ax_idx].transAxes)
            axes[ax_idx].set_title(f'{model}/{family}')
            ax_idx += 1
            continue

        preds = pd.read_csv(pred_file)
        lm = LabelMaker(LabelConfig(target_family=family))
        lm.fit(splits['train'])
        names = lm.class_names()

        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(preds['y_true'], preds['y_pred'])
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

        ax = axes[ax_idx]
        im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
        ax.set_xticks(range(len(names))); ax.set_xticklabels([n[:4] for n in names], fontsize=7)
        ax.set_yticks(range(len(names))); ax.set_yticklabels([n[:4] for n in names], fontsize=7)
        ax.set_title(f'{model}/{family}', fontsize=8)
        for i in range(len(names)):
            for j in range(len(names)):
                ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                        fontsize=7, color='white' if cm_norm[i,j] > 0.5 else 'black')
        ax_idx += 1

plt.suptitle('Confusion matrices (test, row-normalized)', fontsize=11)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved → outputs/reports/confusion_matrices.png')

## 5. Confident-accuracy vs coverage curves

In [ ]:
fig, axes = plt.subplots(1, len(families), figsize=(20, 4))

for ax, family in zip(axes, families):
    lm = LabelMaker(LabelConfig(target_family=family))
    lm.fit(splits['train'])
    n_cls = lm.n_classes()
    ax.axhline(1 / n_cls, ls='--', color='gray', alpha=0.5, label='random')

    for model in models:
        pred_file = pathlib.Path(f'outputs/models/clf/{model}_{family}/{model}_{family}_test_predictions.csv')
        if not pred_file.exists():
            continue
        preds = pd.read_csv(pred_file)
        proba_cols = [c for c in preds.columns if c.startswith('proba_')]
        if not proba_cols:
            continue
        max_p = preds[proba_cols].max(axis=1)
        ths = np.linspace(0.30, 0.95, 25)
        covs, accs = [], []
        for t in ths:
            mask = max_p >= t
            covs.append(mask.mean())
            accs.append((preds.loc[mask,'y_true'] == preds.loc[mask,'y_pred']).mean() if mask.sum() > 5 else np.nan)
        ax.plot(covs, accs, marker='o', markersize=3, label=model)

    ax.set_title(family, fontsize=9)
    ax.set_xlabel('Coverage'); ax.set_ylabel('Accuracy')
    ax.legend(fontsize=7)

plt.suptitle('Confident accuracy vs coverage (test)', fontsize=11)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'coverage_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved → outputs/reports/coverage_curves.png')

## 6. Phase 2: representation comparison (1h-grid vs event bars)

This section runs once `btcusdt_volume_bars.csv` and `btcusdt_dollar_bars.csv` are built by `scripts/build_event_dataset.py`.

In [ ]:
p2_families = p1.sort_values('roc_auc', ascending=False)['target_family'].iloc[:2].tolist() if len(p1) else ['vol_regime']
event_reprs  = ['event_volume', 'event_dollar']

p2 = df_all[
    df_all['representation'].isin(event_reprs) & (df_all['split'] == 'test')
].copy()

if len(p2) == 0:
    print('No event-bar results yet.')
    print('Steps to generate:')
    print('  1. python scripts/fetch_fine_data.py --symbol BTCUSDT --days 90')
    print('  2. python scripts/build_event_dataset.py')
    print('  3. make train-clf data=btcusdt_volume label=vol_regime model=catboost')
    print('  4. make train-clf data=btcusdt_dollar  label=vol_regime model=catboost')
    print('  5. Re-run this notebook')
else:
    comparison = pd.concat([p1, p2]).pivot_table(
        index=['model', 'target_family', 'representation'],
        values=['balanced_acc', 'roc_auc', 'mcc'],
        aggfunc='first'
    ).round(4)
    comparison.to_csv(REPORTS_DIR / 'phase2_representation_leaderboard.csv')
    print('Saved → outputs/reports/phase2_representation_leaderboard.csv')
    print(comparison.to_string())

## 7. Summary and winner selection

In [ ]:
if len(p1) > 0:
    winner = p1.sort_values('roc_auc', ascending=False).iloc[0]
    print('=== PHASE 1 WINNER ===')
    print(f'  Model:         {winner["model"]}')
    print(f'  Target family: {winner["target_family"]}')
    print(f'  Test ROC-AUC:  {winner["roc_auc"]:.4f}')
    print(f'  Test MCC:      {winner["mcc"]:.4f}')
    print(f'  Bal acc:       {winner["balanced_acc"]:.4f} (vs baseline {winner["baseline_bal_acc"]:.4f})')
    print(f'  Trading cov:   {winner["trading_coverage"]:.2f}  hit-rate: {winner["trading_hit_rate"]:.4f}')
    print()
    print('  → Register this as PRD model in experiment_tracking.ipynb')